# Usage demo

This notebook sketches how the two patched tools in this repo (modified
Qiskit SABRE, topology-aware pytket-dqc) are meant to be called, using the
actual parameter names added by the patches — see
[`MODIFICATIONS.md`](../MODIFICATIONS.md) for what each one does.

**Verification status** (see the README's "Reproducing the environment"
section for details):

- Section 1 (Qiskit SABRE) has been run end-to-end against a real build of
  the patched Qiskit and produces the claimed effect (fewer inter-QPU SWAP
  crossings).
- Section 2 (pytket-dqc) has been checked for API correctness (imports,
  gateset requirements, `server_link_capacities`) but **not run
  end-to-end**: distribution requires KaHyPar, and the only
  version pip can install (1.3.7) is incompatible with this pytket-dqc
  version's partitioning code (fails even on pytket-dqc's own unmodified
  test suite). Getting a working KaHyPar (1.3.2, built from source) is a
  separate, environment-specific task — see the README.


## 1. CLA-SABRE routing (Qiskit)

Two toy "QPUs" of 3 qubits each, connected by a single inter-QPU link.
`SabreLayout` is given `qubit_qpu_map` (which QPU each physical qubit
belongs to) and `inter_qpu_coupling_map` (how the QPUs themselves are
connected), which together drive CLA-SABRE's cost-function penalty — see
`MODIFICATIONS.md`'s "CLA-SABRE" section. ((1,10) SABRE instead passes a
`custom_distance_matrix` directly — see that section's "(1,10) SABRE".)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.passes import SabreLayout

# 6 physical qubits: QPU 0 = {0, 1, 2}, QPU 1 = {3, 4, 5}.
# Local coupling within each QPU is all-to-all; qubits 2 and 3 are the
# only physical link crossing QPUs.
local_edges = [(i, j) for i in range(3) for j in range(3) if i != j]
local_edges += [(i + 3, j + 3) for i in range(3) for j in range(3) if i != j]
cross_edges = [(2, 3), (3, 2)]
coupling_map = CouplingMap(local_edges + cross_edges)

qubit_qpu_map = [0, 0, 0, 1, 1, 1]
inter_qpu_coupling_map = [(0, 1)]  # QPU 0 <-> QPU 1

qc = QuantumCircuit(6)
for _ in range(10):
    qc.cx(0, 5)  # deliberately "far" gate, forces routing across QPUs
    qc.cx(1, 4)

sabre_layout = SabreLayout(
    coupling_map,
    seed=0,
    qubit_qpu_map=qubit_qpu_map,
    inter_qpu_coupling_map=inter_qpu_coupling_map,
    # alpha / beta left at their defaults (9.0, 3.0) -- see MODIFICATIONS.md's "CLA-SABRE" section.
)

routed = sabre_layout(qc)
print(routed)


## 2. Superconducting topology-aware distribution (pytket-dqc)

A `NISQNetwork` with `server_link_capacities` set per-edge (rather than
relying on the uniform, per-server `server_ebit_mem`) — see
`MODIFICATIONS.md`'s "pytket-dqc: superconducting hardware adaptation"
section for why that's the meaningful change for superconducting hardware.


In [ ]:
from pytket import Circuit, OpType
from pytket_dqc.networks import NISQNetwork
from pytket_dqc.allocators import HypergraphPartitioning
from pytket_dqc.utils import DQCPass

# Two 2-qubit servers connected by a single link with capacity 2.
network = NISQNetwork(
    server_coupling=[[0, 1]],
    server_qubits={0: [0, 1], 1: [2, 3]},
    server_link_capacities={(0, 1): 2},
)

circ = (
    Circuit(4)
    .add_gate(OpType.CU1, 1.0, [0, 2])
    .H(0)
    .add_gate(OpType.CU1, 1.0, [1, 3])
    .H(1)
    .add_gate(OpType.CU1, 1.0, [0, 2])
    .add_gate(OpType.CU1, 1.0, [1, 3])
)
DQCPass().apply(circ)  # rebases to pytket-dqc's required gateset

distribution = HypergraphPartitioning().allocate(circ, network, seed=0)
print("ebit cost:", distribution.cost())

distributed_circ = distribution.to_pytket_circuit()
print(distributed_circ)
